# Memory Experiment — Kasai Affine-Permutation CSS Codes

High-rate QLDPC codes from Kasai (arXiv:2601.08824) and the hardware-co-designed
instances of Chen Zhao et al. (arXiv:2604.16209).

| preset | n | k |
|---|---|---|
| `chen_p96` | 1152 | 580 |
| `chen_p192` | 2304 | 1156 |
| `chen_p192_d16` | 2304 | 1156 |
| `chen_p384` | 4608 | 2308 |
| `chen_p384_d22` | 4608 | 2308 |
| `kasai_p768` | 9216 | 4612 |

Decoding uses plain BP (`ldpc-bp`, no OSD) through `SimulationPipeline` — the
tier-1 ("T1") stage of the hierarchical decoder in arXiv:2604.16209 — then
chains T1 → T2 (relay-BP) via the pipeline's multi-level decoder support.

## Runtime expectations

Most cells below are **not** interactive. At the default `PRESET = "chen_p96"`,
on 4 cores:

| Section | Runtime |
|---|---|
| Replicate published (n, k) | ~10 s |
| **Build the memory circuit** | **~3 min per build** (tracker GF(2) decomposition) |
| Decode with plain BP (T1) | ~30 min at `MAX_SHOTS = 5000` |
| Hierarchical T1 → T2 chain | hours to days |
| Chen transversal SE block | ~3 min build + decode |

To just verify the path works, run the smoke section below (<1 s); it is also
covered by `tests/test_kasai_code.py::test_kasai_memory_experiment_end_to_end_smoke`.

**Scope:** code construction + syndrome extraction + experiment-level
observables. `KasaiCode` reports `num_logicals` but exposes no explicit logical
operator representatives (`logical_ops_available = False`), so protocols needing
them (transversal gates, lattice surgery) aren't supported for Kasai codes yet.

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.ir.qec_system import QECSystem
from lightstim.noise.config import NoiseConfig
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.kasai_code import (
    KASAI_CODE_PRESETS, KasaiChenExtractionBlock, KasaiCode,
    KasaiCodeExtractionBlock,
)
from lightstim.simulation.decoder_backend import DecoderConfig, SimulationPipeline
from lightstim.simulation.decoder_backend.registry import list_decoders

assert "ldpc-bp" in list_decoders(), "pip install ldpc to register the plain-BP decoder"

## Smoke path (fast — run this first)

Same `KasaiCode → MemoryExperiment → DEM → SimulationPipeline` path as the rest
of the notebook, on a small instance that builds in milliseconds. Nothing below
depends on it.

The template needs $H_X H_Z^T = 0$; multiplier `a = 1` gives translations, which
commute unconditionally, so any offsets yield a valid code.

In [ ]:
t0 = time.perf_counter()

# [[24, 2]] instance — same construction, ~50x smaller than chen_p96.
smoke_code = KasaiCode(P=6, L=4, J=2, f=[(1, 0), (1, 1)], g=[(1, 0), (1, 2)])
assert smoke_code.validate_required_commutativity()
hx, hz = smoke_code.get_css_matrices()
assert np.count_nonzero((hx @ hz.T) % 2) == 0, "CSS commutation must hold"
print(f"n={smoke_code.n_data}  k={smoke_code.num_logicals}  "
      f"rank_x={smoke_code.rank_x}  rank_z={smoke_code.rank_z}")

smoke_system = QECSystem()
smoke_system.add_patch(smoke_code, name="smoke")
smoke_circuit = MemoryExperiment(
    qec_system=smoke_system,
    extraction_block_class=KasaiCodeExtractionBlock,
    rounds=2,
    noise_params=NoiseConfig(p_idle=0.0, p_1q=1e-3, p_2q=1e-3,
                             p_meas=1e-3, p_reset=1e-3),
    noise_model="circuit_level",
    basis="Z",
    z_only=True,
).build()
print(f"qubits={smoke_circuit.num_qubits}  detectors={smoke_circuit.num_detectors}  "
      f"observables={smoke_circuit.num_observables}  "
      f"dem_errors={smoke_circuit.detector_error_model().num_errors}")

smoke_stats = SimulationPipeline(
    decoder_config=DecoderConfig("ldpc-bp", params={"max_iter": 30}),
    max_shots=500, max_errors=10_000, batch_size=250,
    num_workers=1, print_progress=False,
).run(smoke_circuit)
print(f"decoded {smoke_stats.shots} shots, LER={smoke_stats.logical_error_rate:.2e} "
      f"— full path OK in {time.perf_counter() - t0:.2f}s")

## Replicate published (n, k)

Sanity-check every preset against the parameters reported in the papers
(GF(2) ranks of $H_X$, $H_Z$ and the required commuting affine pairs).

In [ ]:
rows = []
for name in sorted(KASAI_CODE_PRESETS):
    code = KasaiCode.from_preset(name)
    preset = KASAI_CODE_PRESETS[name]
    rows.append({"preset": name, "P": code.P, "n": code.n_data,
                 "rank_x": code.rank_x, "rank_z": code.rank_z,
                 "k": code.num_logicals,
                 "expected_n": preset["expected_n"],
                 "expected_k": preset["expected_k"],
                 "commutes": code.validate_required_commutativity()})
df_nk = pd.DataFrame(rows)
assert (df_nk["n"] == df_nk["expected_n"]).all()
assert (df_nk["k"] == df_nk["expected_k"]).all()
assert df_nk["commutes"].all()
df_nk

## Configuration

Defaults follow the circuit-level memory experiment of arXiv:2604.16209:
`rounds = 32`, idling noise **off** (their neutral-atom model neglects it), and
**`z_only = True`** so only Z-ancilla measurements emit detectors.

The z-only detector error model ("$D_Z$" in arXiv:2510.14060) is essential for
plain BP: on the full X+Z DEM, 4-cycles from Y-type errors prevent convergence
entirely at depth, while on $D_Z$ BP converges in a handful of iterations (~98%
of shots at p=1e-3, vs the paper's 98.6%). The backend also merges DEM
mechanisms with identical (detector, observable) footprints — stim leaves X/Y
data-error duplicates unmerged here, and those degenerate columns cost ~7x in BP
convergence. Non-converged shots are heralded as logical errors
(`on_decode_failure="error"`), matching the papers' T1-only accounting.

In [ ]:
PRESET      = "chen_p96"
P_VALUES    = [1e-3]
BASIS       = "Z"          # z_only readout requires Z-basis memory
ROUNDS      = 32
Z_ONLY      = True
MAX_SHOTS   = 5_000        # increase for tighter error bars
MAX_ERRORS  = 100
NUM_WORKERS = 4
BATCH_SIZE  = 25
OUTPUT      = ROOT / "notebooks" / "Memory" / "results" / f"{PRESET}_bp.csv"

BP_PARAMS = {
    "max_iter": 200,
    "bp_method": "minimum_sum",
    "ms_scaling_factor": 0.0,   # 0 = ldpc's dynamic scaling (best convergence)
    "schedule": "serial",       # parallel flooding oscillates on these DEMs
}

## Build the memory circuit

In [ ]:
def build_circuit(preset, p, basis=BASIS, rounds=ROUNDS, z_only=Z_ONLY,
                  noise_model="circuit_level",
                  se_block=KasaiCodeExtractionBlock):
    code = KasaiCode.from_preset(preset)
    system = QECSystem()
    system.add_patch(code, name=preset)
    noise = NoiseConfig(p_idle=0.0, p_1q=p, p_2q=p, p_meas=p, p_reset=p)
    exp = MemoryExperiment(
        qec_system=system,
        extraction_block_class=se_block,
        rounds=rounds,
        noise_params=noise,
        noise_model=noise_model,
        basis=basis,
        z_only=z_only,
    )
    circuit = exp.build()
    return circuit, code

circuit, code = build_circuit(PRESET, P_VALUES[0])
print(f"qubits={circuit.num_qubits}  detectors={circuit.num_detectors}  "
      f"observables={circuit.num_observables}  k={code.num_logicals}")

## Decode with plain BP (tier-1 only)

In [ ]:
pipeline = SimulationPipeline(
    decoder_config=DecoderConfig(
        name="ldpc-bp", backend="cpu", params=BP_PARAMS,
        on_decode_failure="error",
    ),
    max_shots=MAX_SHOTS,
    max_errors=MAX_ERRORS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    print_progress=True,
)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
results = []
for p in P_VALUES:
    circuit, code = build_circuit(PRESET, p)
    task = {"code": PRESET, "p": p, "basis": BASIS, "rounds": ROUNDS,
            "z_only": Z_ONLY, "decoder_name": "ldpc-bp"}
    t0 = time.perf_counter()
    stats = pipeline.run(circuit, task)
    ler = stats.logical_error_rate
    # per-round conversion (eq. 7 of arXiv:2510.14060)
    ler_round = (1 - (1 - 2 * ler) ** (1 / ROUNDS)) / 2 if ler < 0.5 else 0.5
    results.append({**task, "shots": stats.shots, "errors": stats.errors,
                    "ler_shot": ler, "ler_round": ler_round,
                    "seconds": time.perf_counter() - t0})
    print(f"p={p:.2e}: LER/shot={ler:.3e}  LER/round={ler_round:.3e}  "
          f"({stats.errors}/{stats.shots})")

df = pd.DataFrame(results)
df.to_csv(OUTPUT, mode="a", header=not OUTPUT.exists(), index=False)
df

Reference points at `p = 1e-3`, `rounds = 32` (T1-only, heralded):
arXiv:2604.16209 reports **98.6%** tier-1 BP convergence on the [[2304,1156]]
code, i.e. a T1 block error rate of ~1.4% per 32-round shot; `chen_p96` here
lands within statistics of that. The full hierarchical result (with relay-BP
and MIP fallback, which this notebook deliberately omits) is ~5e-7 per shot.

## Hierarchical decoding (T1 → T2 chain)

arXiv:2604.16209 decodes hierarchically: BP (T1) handles almost every shot,
non-converged shots escalate to relay-BP (T2, their Table B1 parameters), and
only T2 failures fall back to integer-programming MLE (T3, omitted here).
Passing a **list** of `DecoderConfig`s builds that chain — each stage re-decodes
only the shots the previous one flagged, and unresolved shots follow the last
stage's `on_decode_failure`.

At 1000 shots (chen_p96, r=32, p=1e-3, z_only): T1 alone heralds 33 non-converged
shots (~3.3%); the chain resolves all of them — 0/1000 errors, none
converged-but-wrong.

In [ ]:
assert "relay-bp" in list_decoders(), 'pip install "relay-bp[stim]" for the T2 stage'

# arXiv:2604.16209 Table B1 relay-BP configuration (T2).
RELAY_PARAMS = {"num_sets": 300, "set_max_iter": 60, "gamma0": 0.1,
                "stop_nconv": 1, "pre_iter": 0}

chain_pipeline = SimulationPipeline(
    decoder_config=[
        DecoderConfig("ldpc-bp", params=BP_PARAMS),
        DecoderConfig("relay-bp", params=RELAY_PARAMS, on_decode_failure="error"),
    ],
    max_shots=MAX_SHOTS,
    max_errors=MAX_ERRORS,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    print_progress=True,
)

circuit, _ = build_circuit(PRESET, P_VALUES[0])
t0 = time.perf_counter()
stats_t12 = chain_pipeline.run(circuit, {"code": PRESET, "p": P_VALUES[0],
                                         "decoder_name": "bp+relay-bp"})
ler_12 = stats_t12.logical_error_rate
ler_12_round = (1 - (1 - 2 * ler_12) ** (1 / ROUNDS)) / 2 if ler_12 < 0.5 else 0.5
print(f"T1+T2 chain: LER/shot={ler_12:.3e}  LER/round={ler_12_round:.3e}  "
      f"({stats_t12.errors}/{stats_t12.shots}, {time.perf_counter()-t0:.0f}s)")

### High-statistics background run (20 T1+T2 residual errors)

The 1000-shot run above shows the chain resolving every T1 escalation but can't
pin down the residual rate itself. Same configuration, run to 20 errors in the
background (~8.1 days, not executed inline):

| shots | errors | LER/shot | wall time |
|---|---|---|---|
| 36,850 | 1 | 2.7e-05 | 17h |
| 99,600 | 4 | 4.0e-05 | 46h |
| 289,800 | 9 | 3.1e-05 | 131h |
| **418,250** | **20 (final)** | **4.78e-05, 95% CI [2.92e-05, 7.39e-05]** | 195h |

The CI brackets arXiv:2604.16209's Table C1 q3 (3e-05) near its lower edge —
so the chain reproduces the paper's post-relay-BP residual even though our T1
escalates ~4x more shots (3.3% vs 0.8%, since we omit their sliding windows).
Per-round (eq. 7 of arXiv:2510.14060): 4.78e-05/shot → 1.49e-06/round.

## Chen transversal SE block

`KasaiChenExtractionBlock` implements the syndrome-extraction schedule of
arXiv:2604.16209 (Sec. 2 / Fig. 2): transversal CNOT layers between ancilla and
data blocks ordered by the code's affine permutations, all J check rows in
parallel, CNOT depth L per basis — same depth as the coloration block. The
constructor validates the paper's co-design condition (every within-run
transition APM commutes with a uniform-orbit reference APM; chen_p96's has 3
length-32 orbits, defining the 3x32 atom layout) and raises for codes lacking
it, e.g. `kasai_p768`.

Both schedules measure the same stabilizers at the same depth, but their LERs
differ: at 1000 shots this block measured **19.0% vs 3.3%** heralded per shot.
The transversal ordering is regular, which correlates hook errors — its DEM has
+19% nnz and ~7x the per-round BP non-convergence (`conv&wrong = 0` for both, so
this is a fault-structure effect, not a decode bug). The paper's own simulations
use coloration ordering; this block models the hardware schedule.

In [ ]:
p0 = P_VALUES[0]
circuit_chen, _ = build_circuit(PRESET, p0, se_block=KasaiChenExtractionBlock)
t0 = time.perf_counter()
stats_chen = pipeline.run(circuit_chen, {"code": PRESET, "p": p0,
                                         "se": "chen_transversal"})
ler_c = stats_chen.logical_error_rate
ler_c_round = (1 - (1 - 2 * ler_c) ** (1 / ROUNDS)) / 2 if ler_c < 0.5 else 0.5
print(f"chen SE:       LER/shot={ler_c:.3e}  LER/round={ler_c_round:.3e}  "
      f"({stats_chen.errors}/{stats_chen.shots}, {time.perf_counter()-t0:.0f}s)")
print(f"coloration SE: LER/shot={df.loc[0, 'ler_shot']:.3e}  "
      f"LER/round={df.loc[0, 'ler_round']:.3e}  "
      f"({df.loc[0, 'errors']}/{df.loc[0, 'shots']})")